# Customer Churn Prediction
## Notebook 3: Model Building

### Objective
Train and compare Logistic Regression and Decision Tree classifiers to predict customer churn.

### Approach
- Load the cleaned dataset.
- Separate features and target.
- Create a stratified train-test split.
- Apply preprocessing through a pipeline to prevent data leakage.
- Train both models using the same split.

Model performance will be evaluated separately in Notebook 4.

In [2]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

PROJECT_DIR = Path("..")
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

df = pd.read_csv(
    PROCESSED_DATA_DIR / "customer_churn_cleaned.csv"
)

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 5625
Testing rows: 1407


## 1. Preprocessing and Model Selection

Numerical features are standardized, while categorical features are one-hot encoded. These transformations are placed inside each model pipeline so they are fitted only on training data.

Two classification algorithms are selected:

- **Logistic Regression:** a linear classification baseline.
- **Decision Tree:** a model that can capture nonlinear relationships and feature interactions.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    )
}

trained_models = {}

## 2. Model Training

Fit each model on the training set using the same preprocessing and data split. The test set remains separate for the final evaluation in Notebook 4.

In [4]:
for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    print(f"{name} trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.


## 3. Save Trained Models

Save the fitted pipelines, including preprocessing and the classifier, so the complete prediction workflow can be loaded later.

In [5]:
import joblib

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, pipeline in trained_models.items():
    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, MODEL_DIR / filename)
    print(f"Saved: {MODEL_DIR / filename}")

Saved: ..\models\logistic_regression.joblib
Saved: ..\models\decision_tree.joblib


## 4. Conclusion

Logistic Regression and Decision Tree classifiers have been trained using a consistent train-test split and preprocessing workflow. The fitted pipelines have been saved for reuse.

No model is selected as the winner at this stage. Performance will be compared in the next notebook using precision, recall, F1-score, ROC-AUC, and confusion matrices.